In [1]:
import gc, importlib.util, json, os
from concurrent.futures import ThreadPoolExecutor

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm

import tensorflow as tf
from huggingface_hub import snapshot_download

import torch
from safetensors.torch import load_file
from transformers import WhisperFeatureExtractor
from transformers.models.qwen3_omni_moe.configuration_qwen3_omni_moe import (
    Qwen3OmniMoeAudioEncoderConfig)
from transformers.models.qwen3_omni_moe.modeling_qwen3_omni_moe import (
    Qwen3OmniMoeAudioEncoder, _get_feat_extract_output_lengths)

SR, WINDOW_SEC = 16000, 2.0
N_SAMPLES = int(SR * WINDOW_SEC)

PADS   = ("zero", "mirror", "repeat", "noise")
PLACES = ("center", "right")
NORMS  = ("peak", "rms", "zscore", "none")

META_CSV   = "data/combine.csv"
RESULT_DIR = "result"
QWEN_PATH  = "/run/media/fourier/Data1/Pras/pretrain_models/Qwen3-Omni-30B-A3B-Thinking"

In [ ]:
class BaseExtractor:
    name = "base"
    dim = None
    def __init__(self, pad="zero", place="center", norm="peak", target_rms=0.05,
                 sr=SR, n_samples=N_SAMPLES, seed=0):
        if pad not in PADS:
            raise ValueError(f"pad must be one of {PADS}, got {pad!r}")
        if place not in PLACES:
            raise ValueError(f"place must be one of {PLACES}, got {place!r}")
        if norm not in NORMS:
            raise ValueError(f"norm must be one of {NORMS}, got {norm!r}")
        self.pad, self.place, self.norm = pad, place, norm
        self.target_rms, self.sr, self.n_samples, self.seed = target_rms, sr, n_samples, seed

    def normalize(self, wav):
        """Set the clip's gain.

        "peak"    divide by max |x|, so the loudest sample is 1.0 (default)
        "rms"     scale to `target_rms`, so clips match in loudness and peaks vary
        "zscore"  zero mean, unit variance (i.e. rms 1.0)
        "none"    leave the recorded gain alone
        """
        wav = np.asarray(wav, dtype=np.float32)
        if self.norm == "none":
            return wav
        if self.norm == "peak":
            scale = float(np.abs(wav).max())
        else:
            scale = float(np.sqrt((wav ** 2).mean()))
            if self.norm == "rms":
                scale /= self.target_rms
        return (wav / scale if scale > 0 else wav).astype(np.float32)

    def load_clip(self, path):
        """16 kHz mono, DC-removed, gain-normalized. No windowing - that is
        `frame_clip`."""
        wav, file_sr = sf.read(path, dtype="float32", always_2d=True)
        wav = wav.mean(axis=1)
        if file_sr != self.sr:
            wav = librosa.resample(wav, orig_sr=file_sr, target_sr=self.sr)
        return self.normalize(wav - wav.mean())

    def frame_clip(self, wav, rng=None):
        """Fit `wav` into an `n_samples` window.

        pad     what surrounds the clip
                "zero"    digital silence
                "mirror"  the clip reflected back and forth
                "repeat"  the clip tiled end to end
                "noise"   gaussian at the clip's own noise floor (room tone)
        place   where the original clip sits
                "center"  filler split evenly on both sides (default)
                "right"   filler all on the right, clip at the start
                Audible only for zero/noise; mirror and repeat fill the window
                either way, so there `place` only shifts the phase.
        """
        n = self.n_samples
        wav = np.asarray(wav, dtype=np.float32)
        if len(wav) >= n:
            return wav[:n]

        gap = n - len(wav)
        left = gap // 2 if self.place == "center" else 0
        right = gap - left
        if self.pad == "mirror":
            return np.pad(wav, (left, right), mode="reflect").astype(np.float32)
        if self.pad == "repeat":
            return np.pad(wav, (left, right), mode="wrap").astype(np.float32)
        out = np.pad(wav, (left, right))
        if self.pad == "noise":
            # Noise floor = RMS of the quietest 10% of 10 ms frames, so the
            # filler sits at the clip's own room tone, not an arbitrary level.
            k = len(wav) // 160 * 160
            sd = 1e-4
            if k >= 160:
                r = np.sqrt((wav[:k].reshape(-1, 160) ** 2).mean(1))
                sd = max(float(np.percentile(r, 10)), 1e-5)
            rng = rng or np.random.default_rng(self.seed)
            out[:left] = rng.normal(0, sd, left)
            if right:
                out[left + len(wav):] = rng.normal(0, sd, right)
        return out.astype(np.float32)

    def prepare(self, path, index=0):
        loaded_wav = self.load_clip(path)
        preprocess_wav = self.frame_clip(loaded_wav, rng=np.random.default_rng([self.seed, index]))
        return preprocess_wav

    def _embed(self, batch):
        raise NotImplementedError

    def close(self):
        """Hand cached GPU blocks back; subclasses drop their model first."""
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            free, total = torch.cuda.mem_get_info()
            print(f"[{self.name}] cuda free {free / 2**30:.1f} / {total / 2**30:.1f} GiB")

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        self.close()

    # --- driver ----------------------------------------------------------
    def extract(self, paths, batch_size=32, num_workers=8, progress=True):
        """Embed every path under this instance's framing. Returns (n, dim).
        """
        paths = list(paths)
        out = np.zeros((len(paths), self.dim), dtype=np.float32)
        desc = f"{self.name}/{self.pad}/{self.place}/{self.norm}"
        pool = ThreadPoolExecutor(num_workers) if num_workers else None
        try:
            for start in tqdm(range(0, len(paths), batch_size), disable=not progress, desc=desc):
                chunk = list(enumerate(paths[start:start + batch_size], start))
                work = lambda item: self.prepare(item[1], item[0])
                batch = list(pool.map(work, chunk)) if pool else [work(i) for i in chunk]
                out[start:start + len(chunk)] = self._embed(batch)
        finally:
            if pool is not None:
                pool.shutdown()
        return out
    
class HearExtractor(BaseExtractor):
    name = "hear"
    dim = 512
    def __init__(self, repo="google/hear", use_gpu=False, **framing):
        super().__init__(**framing)
        if use_gpu:
            # TF claims the whole card up front, which starves the torch model later.
            for gpu in tf.config.list_physical_devices("GPU"):
                try:
                    tf.config.experimental.set_memory_growth(gpu, True)
                except RuntimeError as err:
                    print(f"memory growth already fixed: {err}")
        else:
            print("HEAR on CPU")
            tf.config.set_visible_devices([], "GPU")
        self._model = tf.saved_model.load(snapshot_download(repo))
        self._infer = self._model.signatures["serving_default"]

    def _embed(self, batch):
        out = self._infer(x=tf.constant(np.stack(batch)))["output_0"]
        return np.array(out, dtype=np.float32)

    def close(self):
        self._infer = None
        self._model = None

        tf.keras.backend.clear_session()
        super().close()

class QwenExtractor(BaseExtractor):
    name = "qwen"
    dim = 2048

    @staticmethod
    def _resolve_attn(requested, device, dtype):
        """Pick an attention backend; flash-attn is a bonus, never a requirement."""
        fa2_ok = str(device).startswith("cuda") and dtype in (torch.bfloat16, torch.float16)
        if fa2_ok:
            try:  # installed-but-broken builds only blow up on real import
                importlib.import_module("flash_attn")
            except Exception:
                fa2_ok = False
        if requested in (None, "auto"):
            return "flash_attention_2" if fa2_ok else "sdpa"
        if requested == "flash_attention_2" and not fa2_ok:
            print("flash_attention_2 unavailable here, using sdpa instead")
            return "sdpa"
        return requested

    def __init__(self, model_path=QWEN_PATH, device="cuda", dtype="bfloat16",
                 pooling="mean", attn_implementation="auto", **framing):
        super().__init__(**framing)
        
        self._output_lengths = _get_feat_extract_output_lengths
        self.device, self.dtype, self.pooling = device, getattr(torch, dtype), pooling
        self.attn_implementation = self._resolve_attn(attn_implementation, device, self.dtype)
        self.packs_batches = (self.attn_implementation == "flash_attention_2")

        cfg = json.load(open(os.path.join(model_path, "config.json")))
        audio_cfg = Qwen3OmniMoeAudioEncoderConfig(**cfg["thinker_config"]["audio_config"])
        audio_cfg._attn_implementation = self.attn_implementation
        self.dim = audio_cfg.output_dim
        if pooling == "mean_std":
            self.dim *= 2

        try:
            model = Qwen3OmniMoeAudioEncoder(audio_cfg)
        except (ImportError, ValueError, RuntimeError) as err:
            if self.attn_implementation != "flash_attention_2":
                raise
            print(f"flash_attention_2 init failed ({err}); using sdpa instead")
            self.attn_implementation = "sdpa"
            self.packs_batches = False
            audio_cfg._attn_implementation = "sdpa"
            model = Qwen3OmniMoeAudioEncoder(audio_cfg)

        prefix = "thinker.audio_tower."
        index = json.load(open(os.path.join(model_path, "model.safetensors.index.json")))
        state = {}
        for shard in sorted({v for k, v in index["weight_map"].items() if k.startswith(prefix)}):
            tensors = load_file(os.path.join(model_path, shard))
            state.update({k[len(prefix):]: v for k, v in tensors.items() if k.startswith(prefix)})
            del tensors
        missing, unexpected = model.load_state_dict(state, strict=False, assign=True)
        if missing or unexpected:
            raise RuntimeError(f"audio tower weights off: {missing[:5]} / {unexpected[:5]}")
        del state
        gc.collect()

        self.model = model.to(device=device, dtype=self.dtype).eval()
        self.fe = WhisperFeatureExtractor.from_pretrained(model_path)

    def _embed(self, batch):
        batch = list(batch)
        if not self.packs_batches and len(batch) > 1:
            return np.concatenate([self._embed_packed([w]) for w in batch], axis=0)
        return self._embed_packed(batch)

    def _embed_packed(self, wavs):
        feats = self.fe(wavs, sampling_rate=self.sr, return_attention_mask=True, padding="max_length", return_tensors="pt")
        
        # Whisper pads every clip out to 30 s; keep only the real frames.
        mask = feats["attention_mask"].bool()
        n_frames = int(mask.sum(-1).max())
        mask = mask[:, :n_frames].to(self.device)
        x = feats["input_features"][:, :, :n_frames].to(self.device, self.dtype)
        feature_lens = mask.sum(-1)
        packed = x.permute(0, 2, 1)[mask].permute(1, 0)

        with torch.no_grad():
            hidden = self.model(packed, feature_lens=feature_lens).last_hidden_state
        hidden = hidden.to(torch.float32)

        post_lens = self._output_lengths(feature_lens).tolist()
        if sum(post_lens) != hidden.size(0):
            raise RuntimeError(f"packed length mismatch: {sum(post_lens)} != {hidden.size(0)}")

        embs = []
        for seg in torch.split(hidden, post_lens, dim=0):
            if self.pooling == "mean":
                embs.append(seg.mean(dim=0))
            elif self.pooling == "mean_std":
                embs.append(torch.cat([seg.mean(dim=0), seg.std(dim=0, unbiased=False)]))
            else:
                raise ValueError(f"unsupported pooling: {self.pooling}")
        return torch.stack(embs).cpu().numpy().astype(np.float32)

    def close(self):
        self.model = None
        super().close()

EXTRACTORS = {"hear": HearExtractor, "qwen": QwenExtractor}

In [3]:
df = pd.read_csv("data/combine.csv")

In [ ]:
with EXTRACTORS["hear"](pad="repeat", place="right", norm="peak", use_gpu=True) as ext:
    embeds_hear = ext.extract(df["path_file"].tolist(), batch_size=32, num_workers=24)
    np.save('data/embeds/embeds_hear.npy', embeds_hear)

with EXTRACTORS["qwen"](pad="repeat", place="right", norm="peak", device='cuda') as ext:
    embeds_qwen = ext.extract(df["path_file"].tolist(), batch_size=4, num_workers=24)
    np.save('data/embeds/embeds_qwen.npy', embeds_qwen)